# Imports

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.pipeline import Pipeline


from sklearn.neural_network import MLPClassifier

import warnings
warnings.filterwarnings('ignore')

seed = 1234
np.random.seed(seed)  

# Download data

In [4]:
wine_quality = fetch_ucirepo(id=186)

X = wine_quality.data.features
y = wine_quality.data.targets

# If y is a dataframe, convert to 1D array
y = y.values.ravel()

# Split Dataset

In [ ]:
'''
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed
)

# Then: split temp into train (60%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=seed
)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42
)
# 0.176 of 85% ≈ 15%


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=seed,
    stratify=y
)'''

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=seed
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=seed
)

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=500, random_state=seed))
])

param_grid = {
    'mlp__hidden_layer_sizes': [(32,), (64,), (64, 32), (128, 64)],
    'mlp__alpha': [0.0001, 0.001, 0.01],
    'mlp__learning_rate_init': [0.001, 0.01]
}

#cross validiation 
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=seed
)

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross-Validation Accuracy:", grid.best_score_)

best_model = grid.best_estimator_

y_test_pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Best Parameters: {'mlp__alpha': 0.001, 'mlp__hidden_layer_sizes': (128, 64), 'mlp__learning_rate_init': 0.001}
Best Cross-Validation Accuracy: 0.601691900496039

Test Accuracy: 0.6084615384615385

Classification Report:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         6
           4       0.25      0.09      0.14        43
           5       0.62      0.72      0.67       428
           6       0.63      0.63      0.63       567
           7       0.56      0.49      0.52       216
           8       0.52      0.38      0.44        39
           9       0.00      0.00      0.00         1

    accuracy                           0.61      1300
   macro avg       0.37      0.33      0.34      1300
weighted avg       0.60      0.61      0.60      1300



# Standardize

In [11]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# *optional* Params search

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (64,32)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

grid = GridSearchCV(
    MLPClassifier(max_iter=500, random_state=seed),
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best Validation Score:", grid.best_score_)

Best Params: {'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
Best Validation Score: 0.5730048755452911


# Training and Validation (with pipline and cross val)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    alpha=0.01,
    random_state=seed,
    early_stopping=True,  
    n_iter_no_change=15,
    max_iter=500,
)
'''
# Train
mlp.fit(X_train, y_train)

y_val_pred = mlp.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

y_test_pred = mlp.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))'''

best_model = grid.best_estimator_

y_val_pred = best_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

y_test_pred = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Validation Accuracy: 0.823076923076923
Test Accuracy: 0.6007692307692307

Classification Report:
              precision    recall  f1-score   support

           3       0.33      0.17      0.22         6
           4       0.22      0.16      0.19        43
           5       0.65      0.63      0.64       428
           6       0.63      0.63      0.63       567
           7       0.52      0.61      0.56       216
           8       0.48      0.31      0.38        39
           9       0.00      0.00      0.00         1

    accuracy                           0.60      1300
   macro avg       0.40      0.36      0.37      1300
weighted avg       0.60      0.60      0.60      1300



# stratify, standardize, and gridcv 

In [ ]:
from ucimlrepo import fetch_ucirepo
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

wine_quality = fetch_ucirepo(id=186)

X = wine_quality.data.features
y = wine_quality.data.targets.values.ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=seed
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    alpha=0.001,
    random_state=seed,
    early_stopping=True,
    n_iter_no_change=15,
    max_iter=500
)

# Train
mlp.fit(X_train_scaled, y_train)

y_test_pred = mlp.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
error_rate = 1 - accuracy

print("Test Accuracy:", accuracy)
print("Test Error Rate:", error_rate)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.5784615384615385
Test Error Rate: 0.42153846153846153

Classification Report:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         6
           4       0.17      0.02      0.04        43
           5       0.61      0.67      0.64       428
           6       0.57      0.68      0.62       567
           7       0.55      0.37      0.44       216
           8       0.00      0.00      0.00        39
           9       0.00      0.00      0.00         1

    accuracy                           0.58      1300
   macro avg       0.27      0.25      0.25      1300
weighted avg       0.55      0.58      0.55      1300

